# Sql Essential Statements

Run all cells in a fresh Python3.11 kernel. PostgreSQL runs only in a new temporary directory over a local Unix socket; no API keys, Docker, existing database or administrator privileges. The final regression cell closes the owned connection/server and removes only its temporary data. Original source code and output tables remain in nonexecuting appendices.

In [ ]:
import sys
import subprocess
from pathlib import Path
from tempfile import mkdtemp
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'pgserver==0.1.4', 'psycopg[binary]==3.2.12'], check=True)
import pgserver
import psycopg
from IPython.core.magic import register_cell_magic

data_dir = Path(mkdtemp(prefix='tdb-sql-interview-'))
server = pgserver.get_server(data_dir, cleanup_mode='delete')
connection = psycopg.connect(server.get_uri(), autocommit=True)
assert connection.execute('SHOW listen_addresses').fetchone()[0] == ''
query_results, query_texts, query_history = {}, {}, {}

def print_result(columns, rows):
    values = [['NULL' if value is None else str(value) for value in row] for row in rows]
    widths = [max([len(column)] + [len(row[i]) for row in values]) for i, column in enumerate(columns)]
    def line(row):
        return ' | '.join(value.ljust(width) for value, width in zip(row, widths)).rstrip()
    print(line(columns))
    print('-+-'.join('-' * width for width in widths))
    for row in values:
        print(line(row))
    print('Rows:', len(rows))

@register_cell_magic
def sql(line, cell):
    key = line.strip()
    query_texts[key] = cell.strip()
    history = []
    with connection.cursor() as cursor:
        cursor.execute(cell)
        while True:
            if cursor.description:
                columns, rows = [column.name for column in cursor.description], cursor.fetchall()
                print_result(columns, rows)
                query_results[key] = (columns, rows)
                history.append((columns, rows))
            else:
                print(cursor.statusmessage)
                history.append(cursor.statusmessage)
            if not cursor.nextset():
                break
    query_history[key] = history

print('Python:', sys.version.split()[0], '| PostgreSQL:', connection.execute('SHOW server_version').fetchone()[0])
print('Isolated temporary database; local Unix socket only. SQL magic ready.')

connection.execute("\nCREATE SCHEMA essentials;\nSET search_path TO essentials;\nCREATE TABLE table_name(column1 TEXT, column2 TEXT, column3 INTEGER, condition BOOLEAN);\nINSERT INTO table_name VALUES ('A','x',10,TRUE),('A','y',20,TRUE),('B','z',15,FALSE),('B','w',15,TRUE);\nCREATE TABLE customers(customer_id INTEGER PRIMARY KEY, customer_name TEXT, name TEXT, city TEXT, age INTEGER);\nINSERT INTO customers VALUES (123,'John Smith','John Smith','New York',25),(456,'Jane Doe','Jane Doe','New York',17),(789,'Alex Green','Alex Green','Boston',30);\nCREATE TABLE orders(order_id INTEGER PRIMARY KEY,customer_id INTEGER,product_id INTEGER,quantity INTEGER);\nINSERT INTO orders VALUES (1,123,10,30),(2,123,11,30),(3,456,10,70),(4,999,12,1);\nCREATE TABLE employees(name TEXT,department TEXT,salary INTEGER,city TEXT);\nINSERT INTO employees VALUES ('Bob Johnson','Sales',60000,'New York'),('Samantha Lee','Sales',70000,'New York'),('Maya Chen','Marketing',55000,'Boston'),('Omar Ali','Engineering',80000,'Boston');\nCREATE SCHEMA create_demo;\nCREATE TABLE create_demo.students(id INTEGER PRIMARY KEY,name TEXT,major TEXT,gpa NUMERIC(3,1));\nINSERT INTO create_demo.students VALUES(5678,'Jane Doe','Computer Science',3.5);\nCREATE SCHEMA drop_demo;\nCREATE TABLE drop_demo.table_name(id INTEGER);\nCREATE TABLE drop_demo.indexed_table(id INTEGER);\nCREATE INDEX index_name ON drop_demo.indexed_table(id);\nCREATE VIEW drop_demo.view_name AS SELECT id FROM drop_demo.indexed_table;\nCREATE PROCEDURE drop_demo.procedure_name() LANGUAGE SQL AS 'SELECT 1';\nCREATE SCHEMA alter_demo;\nCREATE TABLE alter_demo.table_name(column1 TEXT,column2 TEXT,obsolete INTEGER);\nINSERT INTO alter_demo.table_name VALUES('A','x',99);\nCREATE INDEX index_name ON alter_demo.table_name(column1);\nCREATE VIEW alter_demo.view_name AS SELECT column1,column2 FROM alter_demo.table_name;\nSET search_path TO essentials;\n")
print("Example schemas and explicit source-matching fixtures ready.")

In [ ]:
%%sql example-0
SELECT column1, column2, column3
FROM table_name
WHERE condition
ORDER BY column1, column2;

In [ ]:
%%sql example-1
SELECT *
FROM customers
WHERE age >= 18
ORDER BY customer_id;

In [ ]:
%%sql example-2
SELECT orders.order_id, customers.customer_name
FROM orders
INNER JOIN customers
ON orders.customer_id = customers.customer_id
ORDER BY customers.customer_id NULLS LAST, orders.order_id;

In [ ]:
%%sql example-3
SELECT customers.customer_name, orders.order_id
FROM customers
LEFT JOIN orders
ON customers.customer_id = orders.customer_id
ORDER BY customers.customer_id NULLS LAST, orders.order_id;

In [ ]:
%%sql example-4
SELECT customers.customer_name, orders.order_id
FROM customers
RIGHT JOIN orders
ON customers.customer_id = orders.customer_id
ORDER BY customers.customer_id NULLS LAST, orders.order_id;

In [ ]:
%%sql example-5
SELECT customers.customer_name, orders.order_id
FROM customers
LEFT OUTER JOIN orders
ON customers.customer_id = orders.customer_id
ORDER BY customers.customer_id NULLS LAST, orders.order_id;

In [ ]:
%%sql example-6
SELECT customers.customer_name, orders.order_id
FROM customers
RIGHT OUTER JOIN orders
ON customers.customer_id = orders.customer_id
ORDER BY customers.customer_id NULLS LAST, orders.order_id;

In [ ]:
%%sql example-7
SELECT name, department, salary
FROM employees
WHERE department = 'Sales' AND salary > 50000
ORDER BY name;

In [ ]:
%%sql example-8
SELECT department, AVG(salary) as avg_salary
FROM employees
GROUP BY department
ORDER BY department;

In [ ]:
%%sql example-10
SELECT customer_id, SUM(quantity) AS total_quantity
FROM orders
GROUP BY customer_id
HAVING SUM(quantity) >= 50
ORDER BY customer_id;

In [ ]:
%%sql example-12
SELECT column1, column2, ROW_NUMBER() OVER (ORDER BY column1, column2) AS row_num
FROM table_name
ORDER BY column1, column2;

In [ ]:
%%sql example-13
SELECT column1, column2, SUM(column3) OVER (PARTITION BY column1) AS column3_sum
FROM table_name
ORDER BY column1, column2;

In [ ]:
%%sql example-14
SELECT column1, column2, RANK() OVER (PARTITION BY column1 ORDER BY column3 DESC) AS rank_num
FROM table_name
ORDER BY column1, column2;

In [ ]:
%%sql example-15
SELECT column1, column2, AVG(column3) OVER (PARTITION BY column1) AS column3_avg
FROM table_name
ORDER BY column1, column2;

In [ ]:
%%sql example-16
SELECT name, city
FROM customers
WHERE city = 'New York'
UNION
SELECT name, city
FROM employees
WHERE city = 'New York'
ORDER BY name;

In [ ]:
%%sql example-18
SET search_path TO create_demo;
CREATE TABLE customers (
  id INT PRIMARY KEY,
  name VARCHAR(50),
  email VARCHAR(100),
  phone VARCHAR(20)
);

In [ ]:
%%sql example-19
INSERT INTO customers (id, name, email, phone)
VALUES (1, 'John Doe', 'johndoe@example.com', '555-555-1234')
RETURNING *;

In [ ]:
%%sql example-20
SELECT * FROM customers;

In [ ]:
%%sql example-22
INSERT INTO students (id, name, major, gpa)
VALUES (1234, 'John Doe', 'Computer Science', 3.5)
RETURNING *;

In [ ]:
%%sql example-24
UPDATE students
SET major = 'Mathematics', gpa = 3.7
WHERE id = 1234
RETURNING *;

In [ ]:
%%sql example-26
DELETE FROM students
WHERE id = 1234;
SELECT * FROM students ORDER BY id;

In [ ]:
%%sql example-28
SET search_path TO drop_demo;
DROP TABLE table_name;

In [ ]:
%%sql example-29
DROP INDEX index_name;

In [ ]:
%%sql example-30
DROP VIEW view_name;

In [ ]:
%%sql example-31
DROP PROCEDURE procedure_name();

In [ ]:
%%sql example-32
SET search_path TO alter_demo;
DROP VIEW view_name;
ALTER TABLE table_name
  ADD COLUMN score INTEGER DEFAULT 0,
  ALTER COLUMN column1 TYPE VARCHAR(80),
  DROP COLUMN obsolete,
  ADD CONSTRAINT score_nonnegative CHECK (score >= 0);
ALTER TABLE table_name DROP CONSTRAINT score_nonnegative;
CREATE VIEW view_name AS SELECT column1, column2 FROM table_name;
SELECT column1, column2, score FROM table_name;

In [ ]:
%%sql example-33
DROP INDEX index_name;
CREATE INDEX index_name ON table_name(column2);
ALTER INDEX index_name RENAME TO renamed_index;
SELECT indexname, indexdef FROM pg_indexes WHERE schemaname = 'alter_demo';

In [ ]:
%%sql example-34
CREATE OR REPLACE VIEW view_name AS
SELECT column1, column2, score FROM table_name;
SELECT * FROM view_name;

In [ ]:
from collections import defaultdict
from decimal import Decimal
assert query_results['example-0'][1] == [('A','x',10),('A','y',20),('B','w',15)]
assert [row[0] for row in query_results['example-1'][1]] == [123,789]
customers = {123:'John Smith',456:'Jane Doe',789:'Alex Green'}
orders = [(1,123,30),(2,123,30),(3,456,70),(4,999,1)]
inner = [(oid,customers[cid]) for oid,cid,_ in orders if cid in customers]
assert query_results['example-2'][1] == inner
left = [(customers[cid],oid) for cid in sorted(customers) for oid in ([oid for oid,customer,_ in orders if customer==cid] or [None])]
right = [(customers.get(cid),oid) for oid,cid,_ in orders]
for key in ['example-3','example-5']:assert query_results[key][1] == left
for key in ['example-4','example-6']:assert query_results[key][1] == right
expected_averages = [('Engineering',Decimal('80000')),('Marketing',Decimal('55000')),('Sales',Decimal('65000'))]
assert query_results['example-8'][1] == expected_averages
totals=defaultdict(int)
for _,cid,qty in orders:totals[cid]+=qty
assert query_results['example-10'][1] == sorted((cid,qty)for cid,qty in totals.items()if qty>=50)
assert [r[-1]for r in query_results['example-12'][1]] == [1,2,3,4]
assert [r[-1]for r in query_results['example-13'][1]] == [30,30,30,30]
assert [r[-1]for r in query_results['example-14'][1]] == [2,1,1,1]
assert [r[-1]for r in query_results['example-15'][1]] == [15,15,15,15]
assert {r[0]for r in query_results['example-16'][1]} == {'John Smith','Jane Doe','Bob Johnson','Samantha Lee'}
assert query_results['example-19'][1] == [(1,'John Doe','johndoe@example.com','555-555-1234')]
assert query_results['example-20'][1] == query_results['example-19'][1]
assert query_results['example-22'][1] == [(1234,'John Doe','Computer Science',Decimal('3.5'))]
assert query_results['example-24'][1] == [(1234,'John Doe','Mathematics',Decimal('3.7'))]
assert query_results['example-26'][1] == [(5678,'Jane Doe','Computer Science',Decimal('3.5'))]
for object_name in ['table_name','index_name','view_name']:
    assert connection.execute('SELECT to_regclass(%s)',('drop_demo.'+object_name,)).fetchone()[0] is None
assert connection.execute("SELECT count(*) FROM pg_proc p JOIN pg_namespace n ON n.oid=p.pronamespace WHERE n.nspname='drop_demo' AND p.proname='procedure_name'").fetchone()[0] == 0
assert query_results['example-32'][1] == [('A','x',0)]
assert query_results['example-34'][1] == [('A','x',0)]
index_name,index_sql = query_results['example-33'][1][0]
assert index_name=='renamed_index' and '(column2)' in index_sql
assert connection.execute("SELECT data_type,character_maximum_length FROM information_schema.columns WHERE table_schema='alter_demo' AND table_name='table_name' AND column_name='column1'").fetchone()==('character varying',80)
assert connection.execute("SELECT count(*) FROM information_schema.columns WHERE table_schema='alter_demo' AND table_name='table_name' AND column_name='obsolete'").fetchone()[0]==0
print('Independent join/filter/group/window/union and DML results passed; all DROP/ALTER objects checked in PostgreSQL catalogs.')
connection.close()
server.cleanup()
assert not data_dir.exists()
print('Owned PostgreSQL server stopped; temporary database data removed.')

## Original source code and transcripts

These exact source blocks are preserved for comparison and are not executed.

### Original block 1

```text
SELECT column1, column2, column3
FROM table_name
WHERE condition;
```

### Original block 2

```text
SELECT *
FROM customers
WHERE age >= 18;
```

### Original block 3

```text
SELECT orders.order_id, customers.customer_name
FROM orders
INNER JOIN customers
ON orders.customer_id = customers.customer_id;
```

### Original block 4

```text
SELECT customers.customer_name, orders.order_id
FROM customers
LEFT JOIN orders
ON customers.customer_id = orders.customer_id;
```

### Original block 5

```text
SELECT customers.customer_name, orders.order_id
FROM customers
RIGHT JOIN orders
ON customers.customer_id = orders.customer_id;
```

### Original block 6

```text
SELECT customers.customer_name, orders.order_id
FROM customers
LEFT OUTER JOIN orders
ON customers.customer_id = orders.customer_id;
```

### Original block 7

```text
SELECT customers.customer_name, orders.order_id
FROM customers
RIGHT OUTER JOIN orders
ON customers.customer_id = orders.customer_id;
```

### Original block 8

```text
SELECT name, department, salary
FROM employees
WHERE department = 'Sales' AND salary > 50000;
```

### Original block 9

```text
SELECT department, AVG(salary) as avg_salary
FROM employees
GROUP BY department;
```

### Original block 10

```text
department | avg_salary
-----------------------
Sales      | 65000
Marketing  | 55000
Engineering| 80000
```

### Original block 11

```text
SELECT customer_id, SUM(quantity) AS total_quantity
FROM orders
GROUP BY customer_id
HAVING SUM(quantity) >= 50;
```

### Original block 12

```text
customer_id | total_quantity
---------------------------
123        | 60
456        | 70
```

### Original block 13

```text
SELECT column1, column2, ..., ROW_NUMBER() OVER (ORDER BY column1) AS row_num
FROM table_name;
```

### Original block 14

```text
SELECT column1, column2, ..., SUM(column3) OVER (PARTITION BY column1) AS column3_sum
FROM table_name;
```

### Original block 15

```text
SELECT column1, column2, ..., RANK() OVER (PARTITION BY column1 ORDER BY column3 DESC) AS rank_num
FROM table_name;
```

### Original block 16

```text
SELECT column1, column2, ..., AVG(column3) OVER (PARTITION BY column1) AS column3_avg
FROM table_name;
```

### Original block 17

```text
SELECT name, city
FROM customers
WHERE city = 'New York'
UNION
SELECT name, city
FROM employees
WHERE city = 'New York';
```

### Original block 18

```text
name        | city
-------------------
John Smith  | New York
Jane Doe    | New York
Bob Johnson | New York
Samantha Lee| New York
```

### Original block 19

```text
CREATE TABLE customers (
  id INT PRIMARY KEY,
  name VARCHAR(50),
  email VARCHAR(100),
  phone VARCHAR(20)
);
```

### Original block 20

```text
INSERT INTO customers (id, name, email, phone)
VALUES (1, 'John Doe', 'johndoe@example.com', '555-555-1234');
```

### Original block 21

```text
SELECT * FROM customers;
```

### Original block 22

```text
id | name     | email                | phone
--------------------------------------------
1  | John Doe | johndoe@example.com | 555-555-1234
```

### Original block 23

```text
INSERT INTO students (id, name, major, gpa)
VALUES (1234, 'John Doe', 'Computer Science', 3.5);
```

### Original block 24

```text
id   | name      | major            | gpa
-----------------------------------------
1234 | John Doe | Computer Science | 3.5
```

### Original block 25

```text
UPDATE students
SET major = 'Mathematics', gpa = 3.7
WHERE id = 1234;
```

### Original block 26

```text
id   | name      | major        | gpa
--------------------------------------
1234 | John Doe | Mathematics | 3.7
```

### Original block 27

```text
DELETE FROM students
WHERE id = 1234;
```

### Original block 28

```text
id   | name      | major        | gpa
--------------------------------------
5678 | Jane Doe | Computer Science | 3.5
```

### Original block 29

```text
DROP TABLE table_name;
```

### Original block 30

```text
DROP INDEX index_name ON table_name;
```

### Original block 31

```text
DROP VIEW view_name;
```

### Original block 32

```text
DROP PROCEDURE procedure_name;
```

### Original block 33

```text
ALTER TABLE table_name
ADD column_name data_type [constraint],
MODIFY column_name data_type [constraint],
DROP column_name,
ADD CONSTRAINT constraint_name constraint_definition,
DROP CONSTRAINT constraint_name;
```

### Original block 34

```text
ALTER INDEX index_name
ADD column_name,
DROP column_name;
```

### Original block 35

```text
ALTER VIEW view_name
AS select_statement;
```